# Libraries and Conf

In [ ]:
# Generate profile report (optional - saved to external HTML to avoid notebook bloat)
users = pd.read_csv("data/sample_features_anonymized.csv")
# To generate full report:
# os.makedirs("reports", exist_ok=True)
# design_report = ProfileReport(users)
# design_report.to_file("reports/user_profile_report.html")


# Utility Functions

In [ ]:
def avg_score_cal(df):
    
    """
    Calculates the average like, reply etc. score of a user
    
    input: takes a dataframe containing tweets of a user
    output: returns the average like, reply, retweet, emojis, sentiment and engagement rate of the tweets from a user
    """
    
    total_tweets = df.shape[0]
    avg_tweet_favorite_count = df.tweet_favorite_count.sum()/total_tweets
    avg_tweet_retweet_count = df.tweet_retweet_count.sum()/total_tweets
    avg_tweet_reply_count = df.tweet_reply_count.sum()/total_tweets
    avg_text_sentiment = df.text_sentiment.sum()/total_tweets
    avg_engagement_rate = df.engagement_rate.sum()/total_tweets
    avg_emojis_count = df.emojis_count.sum()/total_tweets
    avg_tweets_per_day = total_tweets/len(df.day.value_counts().values)

    
    return avg_tweets_per_day, avg_tweet_favorite_count, avg_tweet_retweet_count, avg_tweet_reply_count, avg_text_sentiment, avg_engagement_rate, avg_emojis_count

# https://stackoverflow.com/a/70018607/11105356

def time_range(time):
    
    """
    Calculates the period of the day based on time
    
    input: time of the day as string in hour:minute:second format
    output: returns the period of the day as string 
    
    """
    
    hour = datetime.strptime(time, '%H:%M:%S').hour

    if hour > 20:
      return "Night" # 21-23
    elif hour > 16:
      return "Evening" # 17-20
    elif hour > 11:
      return "Afternoon" # 12-16
    elif hour > 4:
      return "Morning" # 5-11
    else:
      return "Midnight" # 1-4


def day_of_week(day):
    
    """
    Calculates the day of the week based on date
    
    input: date as string in YYYY:MM:DD format
    output: returns the day of the week as string 
    
    """
    dt = datetime.strptime(day, '%Y-%m-%d')
    day_of_week = dt.strftime('%A')
    
    return day_of_week


def most_active_day_cal(df):
    
    """
    Calculates which day of the week the user is most active 
    
    input: takes a dataframe containing tweets of a user
    output: returns the most active day of the week for a user
    """
    df["day_of_week"] = df.day.apply(lambda day: day_of_week(day))
    most_active_day = df.day_of_week.value_counts().idxmax()
    
    return most_active_day
    

def most_active_period_cal(df):
    
    """
    Calculates when the majority of the tweets were posted by a user
    
    input: takes a dataframe containing tweets of a user
    output: returns the most active period of the day for a user
    """
    
    df["time_of_the_day"] = df.time.apply(lambda time: time_range(time))
    most_active_period = df.time_of_the_day.value_counts().idxmax()
    
    return most_active_period



def calculate_avg_personality(df):
    
# df_data = pd.concat([data.drop(columns=['text_personality_score']),
#                      pd.json_normalize(data.text_personality_score.apply(ast.literal_eval))], axis=1).copy()
    personalit_data = pd.json_normalize(df.text_personality_score.apply(ast.literal_eval)).sum()
    avg_personality = dict(personalit_data/df.shape[0])

    return avg_personality['o'], avg_personality['c'], avg_personality['e'], avg_personality['a'], avg_personality['n']


def extract_handles(df):
    
    handles_list = df.text_handles.apply(ast.literal_eval).tolist()
    unique_handles = list(set(list(chain(*handles_list))))
    handles_count = dict(Counter(list(chain(*handles_list))))

    return unique_handles, handles_count


# Feature Extraction

In [ ]:
path = "English/Unzipped"
user_list = pd.DataFrame(columns = ["userfile_name", "user_id", "user_screen_name", "user_followers_count", "user_friends_count", 
                                    "user_statuses_count", "total_handles_mentioned", "possible_impressions", "avg_tweets_per_day", 
                                    "avg_tweet_favorite_count", "avg_tweet_reply_count", "avg_tweet_retweet_count", "avg_text_sentiment", 
                                    "avg_engagement_rate", "avg_emojis_count", "most_active_period", "most_active_day", "diagnosis", "user_group", 
                                    "o", "c", "e", "a", "n"])
for folder in os.listdir(path):
    newPath = path + "/" + folder
    print("Processing ", folder, ".....")
    for fileName in os.listdir(newPath):
        data = pd.read_csv(newPath + "/" + fileName)

        userfile_name = fileName.split('.csv')[0]
        user_id = data['user_id'].iloc[0]
        user_screen_name = data['user_screen_name'].iloc[0]
        user_followers_count = data['user_followers_count'].iloc[0]
        user_friends_count = data['user_friends_count'].iloc[0]
        user_statuses_count = data['user_statuses_count'].iloc[0]
        possible_impressions = data['possible_impressions'].iloc[0]
        diagnosis = folder.split("_")[0]
        
        if diagnosis != 'Control':
            user_group = 'diagnosed'
        else:
            user_group = 'non-diagnosed'
        
        avg_tweets_per_day, avg_tweet_favorite_count, avg_tweet_retweet_count, avg_tweet_reply_count, avg_text_sentiment, avg_engagement_rate, avg_emojis_count = avg_score_cal(data)
        most_active_period = most_active_period_cal(data)
        most_active_day = most_active_day_cal(data)
        
        o, c, e, a, n = calculate_avg_personality(data)
        
        unique_handles, handles_count = extract_handles(data)
        total_handles_mentioned = len(unique_handles)

        # print(userfile_name, user_id, user_screen_name, user_followers_count, user_friends_count, user_statuses_count, possible_impressions, diagnosis) 

        user_list = user_list.append({'userfile_name' : userfile_name, 'user_id' : user_id, 'user_screen_name' : user_screen_name,
                          'user_followers_count' : user_followers_count, 'user_friends_count' : user_friends_count, 
                          'user_statuses_count' : user_statuses_count,  "total_handles_mentioned" : total_handles_mentioned, 
                          'possible_impressions' : possible_impressions, "avg_tweets_per_day" : avg_tweets_per_day,
                          'avg_tweet_favorite_count' : avg_tweet_favorite_count, 'avg_tweet_reply_count': avg_tweet_reply_count, 
                          "avg_tweet_retweet_count" : avg_tweet_retweet_count, 'avg_text_sentiment' : avg_text_sentiment, 
                          'avg_engagement_rate' : avg_engagement_rate, "avg_emojis_count" : avg_emojis_count, 'most_active_period' : most_active_period, 
                          'most_active_day' : most_active_day, 'diagnosis' : diagnosis, "user_group" : user_group,
                           "o" : o, "c" : c, "e" : e, "a" : a, "n" : n }, 
                    ignore_index = True)

user_list.to_csv('English/user_features.csv', index=False)

In [ ]:
user_list

# Mentioned Handles Extraction

In [ ]:
path = "English/Unzipped"

handle_list = pd.DataFrame(columns = ["userfile_name", "user_id", "user_screen_name", "diagnosis", "user_group", 
                                    "unique_handles", "handles_count"])

for folder in os.listdir(path):
    newPath = path + "/" + folder
    print("Processing ", folder, ".....")
    for fileName in os.listdir(newPath):
        data = pd.read_csv(newPath + "/" + fileName)
        
        userfile_name = fileName.split('.csv')[0]
        user_id = data['user_id'].iloc[0]
        user_screen_name = data['user_screen_name'].iloc[0]
        diagnosis = folder.split("_")[0]
        
        if diagnosis != 'Control':
            user_group = 'diagnosed'
        else:
            user_group = 'non-diagnosed'
        
        unique_handles, handles_count = extract_handles(data)
        
        handle_list = handle_list.append({'userfile_name' : userfile_name, 'user_id' : user_id, 'user_screen_name' : user_screen_name,
                          'diagnosis' : diagnosis, "user_group" : user_group,
                          "unique_handles" : unique_handles, "handles_count" : handles_count}, 
                    ignore_index = True)


handle_list.to_csv('English/user_handles.csv', index=False)      

In [ ]:
handle_df = pd.read_csv("English/user_handles.csv")

# print(handle_df.handles_count.apply(ast.literal_eval))
for index, row in handle_df.iterrows():
    handles_dict = ast.literal_eval(row.handles_count)
    # print(handles_dict)
    c = Counter(handles_dict)

    most_interacted = c.most_common(5)
    print("username: ", row.user_screen_name, 
          "most interacted users: ", most_interacted, 
          "total handles mentioned: ", len(row.unique_handles))
    print("----------------------")

# https://stackoverflow.com/questions/40496518/how-to-get-the-3-items-with-the-highest-value-from-dictionary
# https://www.geeksforgeeks.org/python-convert-string-dictionary-to-dictionary/

# Basic Report

user_follower_count -> This variable has a high overall correlation with 4 fields: most_active_period, most_active_day, diagnosis, user_group

possible_impression -> title="This variable has a high overall correlation with 5 fields: user_followers_count, user_friends_count, user_statuses_count, avg_tweets_per_day, avg_engagement_rate"

avg_tweet_fav_count -> title="This variable has a high overall correlation with 2 fields: user_followers_count, avg_tweet_reply_count"

avg_tweet_reply_count -> This variable has a high overall correlation with 1 fields: avg_tweet_favorite_count

avg_engagement_rate -> This variable has a high overall correlation with 4 fields: user_followers_count, user_friends_count, user_statuses_count, possible_impressions







In [ ]:
# Generate profile report (optional - saved to external HTML to avoid notebook bloat)
users = pd.read_csv("data/sample_features_anonymized.csv")
# To generate full report:
# os.makedirs("reports", exist_ok=True)
# design_report = ProfileReport(users)
# design_report.to_file("reports/user_profile_report.html")


# EDA

In [ ]:
import os
data_path = "English/user_features.csv" if os.path.exists("English/user_features.csv") else "data/sample_features_anonymized.csv"
df_user = pd.read_csv(data_path)
df_user

In [ ]:
printmd("**Unique Values By Features**")
for feature in df_user.columns:
    uniq = np.unique(df_user[feature])
    print(feature.ljust(30),len(uniq))

## Dataset Description

**Add a Table for features

In [ ]:
df_user.describe().T

In [ ]:
df_user.describe(include=['object']).T

In [ ]:
numeric_cols = df_user.select_dtypes(include=['float', 'int'])
display(numeric_cols.head())

## Categorical Analysis

In [ ]:
non_numeric_cols = df_user.select_dtypes(include=['object'])
display(non_numeric_cols.head())

printmd("#### Here ``userfile_name`` and ``user_screen_name`` are unique identifiers for users")

### Visualizations

In [ ]:
fig = px.pie(non_numeric_cols['user_group'].value_counts().reset_index().rename(columns={'index':'Type'}), values='user_group', names='Type', title='Distribution of Diagnosed and Non-Diagnosed Group')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(margin=dict(t=50, b=20, l=0, r=0))
fig.show()
printmd("### Distribution is quite Balanced")

In [ ]:
fig = px.pie(non_numeric_cols[non_numeric_cols['user_group']=="diagnosed"]["diagnosis"].value_counts().reset_index().rename(columns={'index':'Type'}), values='diagnosis', names='Type')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(margin=dict(t=50, b=10, l=0, r=0), title='Distribution of Different Mental Health Diagnosis Among the Users ')

fig.show()
printmd("### Distribution is quite imbalanced")

In [ ]:
f = sns.catplot(data= non_numeric_cols, x="most_active_period", col = 'user_group',  kind = 'count', col_wrap = 2)
f.fig.suptitle('Time period of the day when most tweets are posted among the two groups')
f.fig.subplots_adjust(top=0.75,right=0.89)
plt.show()

**Both groups post in midnight then followed by afternoon**

In [ ]:
df_diagnosed = non_numeric_cols[non_numeric_cols['user_group']=="diagnosed"].reset_index(drop=True)

# Create subplots: use 'domain' type for Pie subplot
fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["most_active_period"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["most_active_period"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["most_active_period"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["most_active_period"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["most_active_period"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["most_active_period"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["most_active_period"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["most_active_period"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["most_active_period"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=0.00001, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.35, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=0.00001, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.40, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=0.00001, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.38, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.80, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=10, l=0, r=0), title = 'Tweet posted based on the period of the day among different diagnosed sub-groups')
fig.show()

1. ASD, Anxiety, Bipolar post tweet during midnight and morning
2. ADHD, OCD, PTSD, Eating post tweet during afternoon and midnight
3. Depression post tweet during afternoon and morning
4. Schizophrenia post during morning and evening

In [ ]:
df_diagnosed = non_numeric_cols[non_numeric_cols['user_group']=="diagnosed"].reset_index(drop=True)
df_non_diagnosed = non_numeric_cols[non_numeric_cols['user_group']=="non-diagnosed"].reset_index(drop=True)

# Create subplots: use 'domain' type for Pie subplot
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed['most_active_day'].value_counts().index, 
                     values=df_diagnosed['most_active_day'].value_counts().values, name="Diagnosed"), 1, 1)

fig.add_trace(go.Pie(labels=df_non_diagnosed['most_active_day'].value_counts().index, 
                     values=df_non_diagnosed['most_active_day'].value_counts().values, name="Non-diagnosed"), 1, 2)


# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations 
    annotations=[dict(text='Diagnosed', x=0.001, y=0.01, font_size=20, showarrow=False),
                 dict(text='Non-diagnosed', x=0.60, y=0.01, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=20, l=0, r=0), title = 'Most Active Day of the Week among different Groups')
fig.show()

In [ ]:
df_diagnosed = non_numeric_cols[non_numeric_cols['user_group']=="diagnosed"].reset_index(drop=True)

# Create subplots: use 'domain' type for Pie subplot
fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["most_active_day"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["most_active_day"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["most_active_day"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["most_active_day"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["most_active_day"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["most_active_day"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["most_active_day"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["most_active_day"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["most_active_day"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["most_active_day"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=-0.05, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.27, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=-0.05, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.35, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=-0.05, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.35, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.79, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=0, l=0, r=0), title = 'Most active day of the week among different diagnosed sub-groups')
fig.show()

### Statistical Analysis
#### To measure the relation between two categorical variables, we will perform ``chi-square test`` 

In [ ]:
print(non_numeric_cols.columns)
cat_df = non_numeric_cols.drop(columns=['userfile_name', 'user_screen_name'])
display(cat_df.head())

In [ ]:
# https://medium.com/@ritesh.110587/correlation-between-categorical-variables-63f6bd9bf2f7#:~:text=Chi%2DSquare%20test%20of%20independence,categorical%20variables%20is%20easily%20available.
cat_var1 = ('most_active_period', 'most_active_day', 'diagnosis', 'user_group')
cat_var2 = ('most_active_period', 'most_active_day', 'diagnosis', 'user_group')
## Let us jump to Chi-Square test
## Creating all possible combinations between the above two variables list
cat_var_prod = list(product(cat_var1,cat_var2, repeat = 1))

cat_var_prod

In [ ]:
result = []
for i in cat_var_prod:
    if i[0] != i[1]:
        result.append((i[0],i[1],list(stats.chi2_contingency(pd.crosstab(
                                cat_df[i[0]], cat_df[i[1]])))[1]))
result

In [ ]:
chi_test_output = pd.DataFrame(result, columns = ['cat_var1', 'cat_var2', 
                                                       'coeff'])
## Using pivot function to convert the above DataFrame into a crosstab
display(chi_test_output.pivot(index='cat_var1', columns='cat_var2', values='coeff'))

printmd("""### As we know if p value ≤ 0.05, then there exists a relationship between two variables.  
* ``user_group`` and ``most_active_period`` have a relation
* ``diagnosis`` and ``most_active_period`` have a relation 
* ``most_active_day`` and ``most_active_period`` have a relation""")

## Numerical Analysis

### Tweet Posting Magnitude

In [ ]:
display(df_user['user_statuses_count'].describe())

In [ ]:
# Calculate IQR and bin width using Freedman-Diaconis rule
Q3 = df_user['user_statuses_count'].quantile(0.75)
Q1 = df_user['user_statuses_count'].quantile(0.25)
iqr = Q3 - Q1
bin_width = 2 * iqr / (len(df_user['user_statuses_count']) ** (1/3))

# Calculate number of bins using bin width and min/max values
bin_list = [df_user['user_statuses_count'].min(), Q1+bin_width, Q1+2*bin_width, df_user['user_statuses_count'].max()]
bin_labels = ['low', 'medium', 'high']

# Create bins using pandas cut function
df_user['statuses_frequency'] = pd.cut(df_user['user_statuses_count'], bins=bin_list, labels=bin_labels)

printmd("**Status Frequency Group**")
printmd("Low  (19 - 6921.527) <br> Medium  (6922.527 - 10686.554) <br> High  (10687.554 - 895987) <br>")

display(df_user.statuses_frequency.value_counts())

In [ ]:
f = sns.catplot(data= df_user, x="statuses_frequency", col = 'user_group',  kind = 'count', col_wrap = 2)
f.fig.suptitle('Relation between Users" Diagnosis and Tweeting Magnitude')
f.fig.subplots_adjust(top=0.81,right=0.86)
plt.show()

printmd("#### Tweeting Magnitude is similar in both groups")

In [ ]:
df_diagnosed = df_user[df_user['user_group']=="diagnosed"].reset_index(drop=True)

fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["statuses_frequency"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["statuses_frequency"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["statuses_frequency"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["statuses_frequency"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["statuses_frequency"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["statuses_frequency"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["statuses_frequency"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["statuses_frequency"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["statuses_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["statuses_frequency"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=-0.05, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.27, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=-0.05, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.35, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=-0.05, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.35, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.79, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=0, l=0, r=0), title = 'Tweet posting tendency among different diagnosed sub-groups')
fig.show()


printmd("#### Eating, Asd, Schizophrenia have low tweeting tendency, whereas, ADHD, Bipolar, OCD and Anxiety have high tweeting tendency, Depression and Ptsd are similar with both high and low tendency")

### Mentioning Other Users

In [ ]:
display(df_user['total_handles_mentioned'].describe())

In [ ]:
# Calculate IQR and bin width using Freedman-Diaconis rule
Q3 = df_user['total_handles_mentioned'].quantile(0.75)
Q1 = df_user['total_handles_mentioned'].quantile(0.25)
iqr = Q3 - Q1
bin_width = 2 * iqr / (len(df_user['total_handles_mentioned']) ** (1/3))

# Calculate number of bins using bin width and min/max values
bin_list = [df_user['total_handles_mentioned'].min(), Q1+bin_width, Q1+2*bin_width, df_user['total_handles_mentioned'].max()]
bin_labels = ['low', 'medium', 'high']

# Create bins using pandas cut function
df_user['mention_frequency'] = pd.cut(df_user['total_handles_mentioned'], bins=bin_list, labels=bin_labels)

printmd("**User Mention Frequency Group**")
printmd("Low  (0 - 149) <br> Medium  (150 - 218) <br> High  (219 - 3590) <br>")

display(df_user.mention_frequency.value_counts())

In [ ]:
f = sns.catplot(data= df_user, x="mention_frequency", col = 'user_group',  kind = 'count', col_wrap = 2)
f.fig.suptitle('Relation between Users" Mentiong Other Users and Their Diagnosis')
f.fig.subplots_adjust(top=0.81,right=0.86)
plt.show()

printmd("#### Mentioning other users' is similar in both groups")

In [ ]:
df_diagnosed = df_user[df_user['user_group']=="diagnosed"].reset_index(drop=True)

fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["mention_frequency"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["mention_frequency"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["mention_frequency"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["mention_frequency"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["mention_frequency"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["mention_frequency"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["mention_frequency"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["mention_frequency"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["mention_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["mention_frequency"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=-0.05, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.27, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=-0.05, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.35, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=-0.05, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.35, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.79, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=0, l=0, r=0), title = 'Tendency to mention other users" in tweet among different diagnosed sub-groups')
fig.show()


printmd("#### People who are diagnosed with Schizophrenia have lower tendency to mention other users")

### User Friends Count

In [ ]:
display(df_user['user_friends_count'].describe())

In [ ]:
# Calculate IQR and bin width using Freedman-Diaconis rule
Q3 = df_user['user_friends_count'].quantile(0.75)
Q1 = df_user['user_friends_count'].quantile(0.25)
iqr = Q3 - Q1
bin_width = 2 * iqr / (len(df_user['user_friends_count']) ** (1/3))

# Calculate number of bins using bin width and min/max values
bin_list = [df_user['user_friends_count'].min(), Q1+bin_width, Q1+2*bin_width, df_user['user_friends_count'].max()]
bin_labels = ['low', 'medium', 'high']

# Create bins using pandas cut function
df_user['friendship_frequency'] = pd.cut(df_user['user_friends_count'], bins=bin_list, labels=bin_labels)

printmd("**User's Friend Circle**")
printmd("Low  (0 - 390) <br> Medium  (340 - 537) <br> High  (538 - 104314) <br>")

display(df_user.friendship_frequency.value_counts())

In [ ]:
f = sns.catplot(data= df_user, x="friendship_frequency", col = 'user_group',  kind = 'count', col_wrap = 2)
f.fig.suptitle('Relation between User dignosed groups and their magnitude of friends')
f.fig.subplots_adjust(top=0.81,right=0.86)
plt.show()

printmd("#### Magnitude of friend's is quite similar in both groups")

In [ ]:
df_diagnosed = df_user[df_user['user_group']=="diagnosed"].reset_index(drop=True)

fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["friendship_frequency"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["friendship_frequency"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["friendship_frequency"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["friendship_frequency"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["friendship_frequency"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["friendship_frequency"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["friendship_frequency"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["friendship_frequency"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["friendship_frequency"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["friendship_frequency"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=-0.05, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.27, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=-0.05, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.35, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=-0.05, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.35, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.79, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=0, l=0, r=0), title = 'Magnitude of Friend-cirle among different diagnosed sub-groups')
fig.show()


printmd("#### People who are diagnosed with Eating Disorder have lower number of friends")

### Average Tweet Sentiment

In [ ]:
df_user.avg_text_sentiment.describe()

In [ ]:
min_val = df_user.avg_text_sentiment.min()
max_val = df_user.avg_text_sentiment.max()

# Divide the range into three bins
bin_list = [min_val, 0.0, 0.29101975, max_val]

bin_labels = ['negative', 'Neutral', 'Positive']

# Create bins using pandas cut function
df_user['tweet_sentiment_range'] = pd.cut(df_user['avg_text_sentiment'], bins=bin_list, labels=bin_labels)

printmd("**Tweet Sentiment Range**")
printmd("Negative  (-0.511 - 0.0) <br> Neutral  (0.0 - 0.291) <br> Positive  (0.291 - 0.6922) <br>")

display(df_user.tweet_sentiment_range.value_counts())

In [ ]:
f = sns.catplot(data= df_user, x="tweet_sentiment_range", col = 'user_group',  kind = 'count', col_wrap = 2)
f.fig.suptitle('Relation between User dignosed groups and their tweet sentiment')
f.fig.subplots_adjust(top=0.81,right=0.86)
plt.show()

printmd("#### Tweet sentiment range is quite similar in both groups with a neural tome. But non-diagnosed group posts more positive tweets")

In [ ]:
df_diagnosed = df_user[df_user['user_group']=="diagnosed"].reset_index(drop=True)

fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Adhd"]["tweet_sentiment_range"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Depression"]["tweet_sentiment_range"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Asd"]["tweet_sentiment_range"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Bipolar"]["tweet_sentiment_range"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ptsd"]["tweet_sentiment_range"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Anxiety"]["tweet_sentiment_range"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Ocd"]["tweet_sentiment_range"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Eating"]["tweet_sentiment_range"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["tweet_sentiment_range"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['diagnosis']=="Schizophrenia"]["tweet_sentiment_range"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=-0.05, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.27, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=-0.05, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.35, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=-0.05, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.35, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.79, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=0, l=0, r=0), title = 'Tweet sentiment range among different diagnosed sub-groups')
fig.show()


printmd("#### People who are diagnosed with Eating Disorder, PTSD and Schizophrenia have more negative tweets")

### Correlation with User Group

In [ ]:
le = LabelEncoder()
df_user['user_group_encoded'] = le.fit_transform(df_user['user_group'])
display(df_user.user_group_encoded.value_counts())


In [ ]:
def measure_correlation(df):
    df = df[['user_followers_count', 'user_friends_count',
           'user_statuses_count', 'total_handles_mentioned',
           'possible_impressions', 'avg_tweets_per_day',
           'avg_tweet_favorite_count', 'avg_tweet_reply_count',
           'avg_tweet_retweet_count', 'avg_text_sentiment', 'avg_engagement_rate',
           'avg_emojis_count', 'o', 'c', 'e', 'a', 'n', 'user_group_encoded']]

    columns_list = list(df.columns)[:-1]
    
    for column in columns_list:  
        scaler = StandardScaler()
        df[column] = scaler.fit_transform(df[[column]])
        x = df[column]
        y = df['user_group_encoded']

        # calculate point-biserial correlation coefficient and p-value
        corr, p_value = pointbiserialr(x, y)
        # print the results
        print("Correlation between ", column, " and user dignosed group")
        if p_value < 0.05:
            print("Point-biserial correlation coefficient: ", corr)
            print("p-value is significant: ", p_value)
        else:
            print("Not Significant: ", p_value)
        print("--------------------------------------------------------")
measure_correlation(df_user)

**user_friends_count, avg_text_sentiment, o, c, n are kind of correlated to user group**

# Clustering

In [ ]:
sliced_data = df_user[['n', 'c', 'user_group_encoded', 'diagnosis']]
      
print((sliced_data.user_group_encoded.value_counts()))
print(df_user.user_group.value_counts())

In [ ]:
def plot_clusters_in_group(df):
    
# Group by category and count the occurrences of each value
    counts = df.groupby('diagnosis')['clusters'].value_counts()

# Convert the counts to a dataframe and reset the index
    counts_df = counts.to_frame(name='Count').reset_index()

# Plot a bar chart for each category
    fig = plt.figure(figsize=(8, 5))
    for category in df['diagnosis'].unique():
        category_counts = counts_df[counts_df['diagnosis'] == category]
        plt.bar(category_counts['clusters'], category_counts['Count'], label=category)

# Add labels and legend
    plt.xlabel('clusters')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

## Agglomerative Clustering

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler


# Separate continuous and binary variables
def agglomerative_clustering(data):
    X_continuous = data[['n', 'c']].values
    X_binary =  data['user_group_encoded'].values

    # Standardize the continuous variables
    scaler = StandardScaler()
    X_continuous = scaler.fit_transform(X_continuous)

    # Concatenate the two arrays
    X = np.concatenate((X_continuous, X_binary.reshape(-1, 1)), axis=1)

    # Apply agglomerative clustering
    model = AgglomerativeClustering(n_clusters=10)
    clusters = model.fit_predict(X)

    # Add cluster labels to dataframe
    data['clusters'] = clusters

    # Print cluster assignments
    # print(user[['n', 'o', 'user_group_encoded', 'clusters']])

    # Plot the clusters
    
    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(X[:, 0], X[:, 1], c=clusters)
    legend1 = ax.legend(*scatter.legend_elements(), title='Cluster')
    ax.add_artist(legend1)
    ax.set_xlabel('Neuroticism')
    ax.set_ylabel('Conscientiousness')
    ax.set_title('Agglomerative Clustering')
    plt.show()
    
    return data

data_agg = agglomerative_clustering(sliced_data)

plot_clusters_in_group(data_agg)

## DBSCAN

In [ ]:
def dbscan_clustering(data):
    # Separate continuous and binary variables
    X_continuous = data[['n', 'c']].values
    X_binary =  data['user_group_encoded'].values

    # Standardize the continuous variables
    scaler = StandardScaler()
    X_continuous = scaler.fit_transform(X_continuous)

    # Concatenate the two arrays
    X = np.concatenate((X_continuous, X_binary.reshape(-1, 1)), axis=1)

    labels_true = X_binary

    # label_list = list(X_binary)

    model = DBSCAN(eps=0.35, min_samples=10)
    # fit model and predict clusters
    yhat = model.fit_predict(X)

    labels = model.labels_
    
    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)
    
    data['clusters'] = yhat
    
    print("Estimated number of clusters: %d" % n_clusters_)
    print("Estimated number of noise points: %d" % n_noise_)

    print(f"Homogeneity: {metrics.homogeneity_score(labels_true, labels):.3f}")
    print(f"Completeness: {metrics.completeness_score(labels_true, labels):.3f}")
    print(f"V-measure: {metrics.v_measure_score(labels_true, labels):.3f}")
    print(f"Adjusted Rand Index: {metrics.adjusted_rand_score(labels_true, labels):.3f}")
    print(
    "Adjusted Mutual Information:"
    f" {metrics.adjusted_mutual_info_score(labels_true, labels):.3f}"
    )
    print(f"Silhouette Coefficient: {metrics.silhouette_score(X, labels):.3f}")
    
    
    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(X[:, 0], X[:, 1], c=yhat)
    legend1 = ax.legend(*scatter.legend_elements(), title='Cluster')
    ax.add_artist(legend1)
    ax.set_xlabel('Neuroticism')
    ax.set_ylabel('Conscientiousness')
    ax.set_title('DBSCAN')
    plt.show()
    
    return data

data_dbscan = dbscan_clustering(sliced_data)
plot_clusters_in_group(data_dbscan)

## KMeans

In [ ]:
def kmeans_clustering(data):
    # Separate continuous and binary variables
    X_continuous = data[['n', 'c']].values
    X_binary =  data['user_group_encoded'].values

    # Standardize the continuous variables
    scaler = StandardScaler()
    X_continuous = scaler.fit_transform(X_continuous)
    X = np.concatenate((X_continuous, X_binary.reshape(-1, 1)), axis=1)
    
    # apply KMeans clustering
    kmeans = KMeans(n_clusters=10, random_state=42)
    kmeans.fit(X)

    # get cluster labels
    y_kmeans = kmeans.predict(X)
    data['clusters'] = y_kmeans
    
    # Plot the clusters
    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y_kmeans)
    legend1 = ax.legend(*scatter.legend_elements(), title='Cluster')
    ax.add_artist(legend1)
    ax.set_xlabel('Neuroticism')
    ax.set_ylabel('Conscientiousness')
    ax.set_title('KMeans')
    plt.show()
    
    return data

data_kmeans = kmeans_clustering(sliced_data)
plot_clusters_in_group(data_kmeans)

# Network Analysis

In [ ]:
import os
net_path = "English/network_features.csv" if os.path.exists("English/network_features.csv") else "data/sample_features_anonymized.csv"
df_network = pd.read_csv(net_path)
df_network

## Selecting User Sample

In [ ]:
network_of_list = list(df_network.network_of)
counts = Counter(network_of_list)

# get unique users
unique_users = list(counts.keys())
print(len(unique_users))

In [ ]:
selected_users = df_user[df_user['userfile_name'].isin(unique_users)]
selected_users

## Adding Network Diagnose Info

In [ ]:
df_network = pd.merge(df_network, selected_users[['userfile_name', 'diagnosis']], left_on='network_of', right_on='userfile_name', how='left')
df_network.rename(columns={'diagnosis': 'network_diagnose'}, inplace=True)
df_network.rename(columns={'userfile_name_x': 'userfile_name'}, inplace=True)
df_network = df_network.drop('userfile_name_y', axis=1)
df_network

## Network Dataset Description

In [ ]:
df_network.describe().T

In [ ]:
df_network.describe(include=['object']).T

In [ ]:
numeric_cols = df_network.select_dtypes(include=['float', 'int'])
display(numeric_cols.head())

In [ ]:
non_numeric_cols = df_network.select_dtypes(include=['object'])
display(non_numeric_cols.head())

printmd("#### Here ``userfile_name`` and ``user_screen_name`` are unique identifiers for users")

## Visualizations

In [ ]:
fig = px.pie(non_numeric_cols['user_group'].value_counts().reset_index().rename(columns={'index':'Type'}), values='user_group', names='Type', title='Distribution of Diagnosed and Non-Diagnosed Network Group')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(margin=dict(t=50, b=20, l=0, r=0))
fig.show()
printmd("### Distribution is quite Balanced")

In [ ]:
f = sns.catplot(data= non_numeric_cols, x="most_active_period", col = 'user_group',  kind = 'count', col_wrap = 2)
f.fig.suptitle('Time period of the day when most tweets are posted among the two network groups')
f.fig.subplots_adjust(top=0.75,right=0.89)
plt.show()

1. For control network, mostly post in afternoon then followed by midnight
2. For diagnosed network, mostly post in midnight then followed by afternoon

In [ ]:
df_diagnosed = non_numeric_cols[non_numeric_cols['user_group']=="diagnosed"].reset_index(drop=True)

# Create subplots: use 'domain' type for Pie subplot
fig = make_subplots(rows=3, cols=3, specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}], 
                                          [{'type':'domain'}, {'type':'domain'}, {'type':'domain'}]])

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Adhd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Adhd"]["most_active_period"].value_counts().values, name="Adhd"), 1, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Depression"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Depression"]["most_active_period"].value_counts().values, name="Depression"), 1, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Asd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Asd"]["most_active_period"].value_counts().values, name="Asd"), 1, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Bipolar"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Bipolar"]["most_active_period"].value_counts().values, name="Bipolar"), 2, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Ptsd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Ptsd"]["most_active_period"].value_counts().values, name="Ptsd"), 2, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Anxiety"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Anxiety"]["most_active_period"].value_counts().values, name="Anxiety"), 2, 3)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Ocd"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Ocd"]["most_active_period"].value_counts().values, name="Ocd"), 3, 1)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Eating"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Eating"]["most_active_period"].value_counts().values, name="Eating"), 3, 2)

fig.add_trace(go.Pie(labels=df_diagnosed[df_diagnosed['network_diagnose']=="Schizophrenia"]["most_active_period"].value_counts().index, 
                     values=df_diagnosed[df_diagnosed['network_diagnose']=="Schizophrenia"]["most_active_period"].value_counts().values, name="Schizophrenia"), 3, 3)

# donut-like pie chart
# fig.update_traces(hoverinfo="label+percent")

fig.update_layout(

    # Add annotations
    annotations=[dict(text='Adhd', x=0.00001, y=0.90, font_size=20, showarrow=False),
                 dict(text='Depression', x=0.35, y=0.90, font_size=20, showarrow=False),
                 dict(text='Asd', x=0.73, y=0.90, font_size=20, showarrow=False),
                 dict(text='Bipolar', x=0.00001, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ptsd', x=0.40, y=0.55, font_size=20, showarrow=False),
                 dict(text='Anxiety', x=0.75, y=0.55, font_size=20, showarrow=False),
                 dict(text='Ocd', x=0.00001, y=0.10, font_size=20, showarrow=False),
                 dict(text='Eating', x=0.38, y=0.10, font_size=20, showarrow=False),
                 dict(text='Schizophrenia', x=0.75, y=0.10, font_size=20, showarrow=False)])
fig.update_layout(margin=dict(t=50, b=10, l=0, r=0), title = 'Tweet posted based on the period of the day among different diagnosed network sub-groups')
fig.show()

Users
1. ASD, Anxiety, Bipolar post tweet during midnight and morning
2. ADHD, OCD, PTSD, Eating post tweet during afternoon and midnight
3. Depression post tweet during afternoon and morning
4. Schizophrenia post during morning and evening

Network
1. ADHD, ASD, PTSD evening and midnight
2. Depression, Anxiety, afternoon and midnight
3. Bipolar evening and afternoon
4. OCD evening and morning
5. Eating morning and midnight
6. Schizophrenia midnight